# Predict-and-Save: AI-Driven Retention Engine
**Company A · Business Proposal · Kaileshwar · GCI 2026**

Turning 100,000 customer records into ₹3.79 bn of annual net benefit

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix, average_precision_score,
                             roc_curve, precision_recall_curve)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.figsize": (12, 6)})
print("Libraries loaded")

## 1. Data Loading & Merging
`Client.csv` (demographics, 50 cols) + `Record.csv` (usage, 51 cols) → 100K × 100

In [ ]:
df_client = pd.read_csv("Client.csv", low_memory=False)
df_record = pd.read_csv("Record.csv", low_memory=False)
print(f"Client: {df_client.shape} | Record: {df_record.shape}")

df = pd.merge(df_client, df_record, on="Customer_ID", how="inner")
print(f"Merged: {df.shape}")
print(f"Categorical: {df.select_dtypes(include=['object','category']).shape[1]} | Numeric: {df.select_dtypes(include=[np.number]).shape[1]}")
df.head()

## 2. Data Quality & Missingness
Some demographic fields >35% NaN — XGBoost handles missing natively (no imputation needed)

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Count": missing, "Pct": missing_pct})
missing_df = missing_df[missing_df["Count"] > 0].sort_values("Pct", ascending=False)
print(f"Columns with missing: {len(missing_df)} / {df.shape[1]}")
print(f"Overall missing: {missing.sum() / df.size * 100:.2f}%")
print(f"\nTop missing (tree handles natively):")
missing_df.head(10)

## 3. Churn Distribution
Target balance: ~49.6% churn / ~50.4% stay — accuracy and F1 both meaningful

In [ ]:
churn_rate = df["churn"].mean() * 100
cc = df["churn"].value_counts()
print(f"Churn Rate: {churn_rate:.2f}% | Churned: {cc.get(1,0):,} | Retained: {cc.get(0,0):,}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.pie([cc.get(0,0), cc.get(1,0)], labels=["Retained","Churned"], colors=["#457B9D","#E63946"],
        autopct="%1.1f%%", shadow=True, startangle=140, explode=(0, 0.08))
ax1.set_title("Churn Distribution", fontweight="bold")

bins = pd.cut(df["months"], bins=[0,6,12,24,36,100], labels=["0-6m","6-12m","12-24m","24-36m","36m+"])
ct = df.groupby(bins, observed=False)["churn"].mean() * 100
ax2.bar(ct.index.astype(str), ct.values, color=["#E63946" if v>churn_rate else "#457B9D" for v in ct.values])
ax2.axhline(churn_rate, color="black", ls="--", label=f"Avg ({churn_rate:.1f}%)")
ax2.set_xlabel("Tenure"); ax2.set_ylabel("Churn Rate (%)"); ax2.set_title("Churn by Tenure", fontweight="bold")
ax2.legend()
plt.tight_layout()
plt.show()

## 4. EDA: Equipment Age & Tenure — Two Intervenable Levers
Customers with 24+ month-old handsets churn at higher rates; mid-tenure (12–18 mo) is the danger zone

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Equipment age vs churn
df["eqp_bin"] = pd.cut(df["eqpdays"], bins=[0, 180, 365, 540, 730, 2000],
                       labels=["0-6mo","6-12mo","12-18mo","18-24mo","24mo+"])
eqp_churn = df.groupby("eqp_bin", observed=False)["churn"].mean() * 100
bars = ax1.bar(eqp_churn.index.astype(str), eqp_churn.values,
               color=["#E63946" if v > churn_rate else "#457B9D" for v in eqp_churn.values])
ax1.axhline(churn_rate, color="black", ls="--", label=f"Avg ({churn_rate:.1f}%)")
ax1.set_xlabel("Equipment Age"); ax1.set_ylabel("Churn Rate (%)")
ax1.set_title("Churn by Equipment Age", fontweight="bold"); ax1.legend()
for b, v in zip(bars, eqp_churn.values):
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f"{v:.1f}%", ha="center", fontsize=9)

# Tenure danger zone
tenure_bins = pd.cut(df["months"], bins=[0,6,12,18,24,36,100],
                     labels=["0-6","6-12","12-18","18-24","24-36","36+"])
ten_churn = df.groupby(tenure_bins, observed=False)["churn"].mean() * 100
bars2 = ax2.bar(ten_churn.index.astype(str), ten_churn.values,
                color=["#E63946" if v > churn_rate else "#457B9D" for v in ten_churn.values])
ax2.axhline(churn_rate, color="black", ls="--", label=f"Avg ({churn_rate:.1f}%)")
ax2.set_xlabel("Tenure (months)"); ax2.set_ylabel("Churn Rate (%)")
ax2.set_title("Churn by Tenure — Danger Zone", fontweight="bold"); ax2.legend()
for b, v in zip(bars2, ten_churn.values):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 5. EDA: Quiet Customers Are Dangerous
Usage drop predicts churn; customers who never call support churn most (silent dissatisfaction)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Usage change vs churn
df["mou_change_bin"] = pd.cut(df["change_mou"], bins=[-np.inf, -200, -50, 0, 50, 200, np.inf],
                              labels=["<-200","-200 to -50","-50 to 0","0 to 50","50 to 200",">200"])
mou_churn = df.groupby("mou_change_bin", observed=False)["churn"].mean() * 100
bars = ax1.bar(mou_churn.index.astype(str), mou_churn.values,
               color=["#E63946" if v > churn_rate else "#457B9D" for v in mou_churn.values])
ax1.axhline(churn_rate, color="black", ls="--", label=f"Baseline ({churn_rate:.1f}%)")
ax1.set_xlabel("MOU Change"); ax1.set_ylabel("Churn Rate (%)")
ax1.set_title("Churn by Usage Change (MOU)", fontweight="bold"); ax1.legend()
ax1.tick_params(axis='x', rotation=30)

# Customer care calls vs churn
df["cc_bin"] = pd.cut(df["custcare_Mean"], bins=[-0.1, 0, 0.5, 1.5, 3, 100],
                      labels=["0 calls","0-0.5","0.5-1.5","1.5-3","3+"])
cc_churn = df.groupby("cc_bin", observed=False)["churn"].mean() * 100
bars2 = ax2.bar(cc_churn.index.astype(str), cc_churn.values,
                color=["#E63946" if v > churn_rate else "#457B9D" for v in cc_churn.values])
ax2.axhline(churn_rate, color="black", ls="--", label=f"Baseline ({churn_rate:.1f}%)")
ax2.set_xlabel("Avg Monthly Care Calls"); ax2.set_ylabel("Churn Rate (%)")
ax2.set_title("Silent Customers Churn Most", fontweight="bold"); ax2.legend()

plt.tight_layout()
plt.show()

## 6. Revenue × Usage Trend Heatmap

In [ ]:
df["rev_trend"] = pd.cut(df["change_rev"], bins=[-np.inf,-5,5,np.inf], labels=["Declining","Stable","Growing"])
df["usage_trend"] = pd.cut(df["change_mou"], bins=[-np.inf,-10,10,np.inf], labels=["Declining","Stable","Growing"])
pivot = df.groupby(["rev_trend","usage_trend"], observed=False)["churn"].mean().unstack()*100

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r", linewidths=0.5,
            cbar_kws={"label":"Churn Rate (%)"}, annot_kws={"size":14,"weight":"bold"}, ax=ax)
ax.set_title("Churn: Revenue × Usage Trend", fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Feature Correlation with Churn

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
churn_corr = df[num_cols].corr()["churn"].drop("churn").abs().sort_values(ascending=False)
top15 = churn_corr.head(15).index.tolist()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[["churn"]+top15].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.5, annot_kws={"size":8}, ax=ax)
ax.set_title("Top 15 Correlations with Churn", fontweight="bold")
plt.tight_layout()
plt.show()

print("\nTop 10 churn correlates:")
for i,(f,c) in enumerate(churn_corr.head(10).items()):
    print(f"  {i+1}. {f}: {c:.4f}")

## 8. ML Pipeline
**70/30 stratified split** · XGBoost: 500 trees, depth 6, LR 0.05 · 5-fold CV hyperparameters

In [ ]:
cat_cols = df.select_dtypes(include=["object","category"]).columns.tolist()
num_ml = [c for c in num_cols if c not in ["churn","Customer_ID"]]
high_miss = [c for c in df.columns if df[c].isnull().mean()>0.30]
num_ml = [c for c in num_ml if c not in high_miss]

df_ml = df.copy()
for col in cat_cols:
    if col in high_miss: continue
    df_ml[col] = df_ml[col].astype(str).replace("nan","MISSING")
    df_ml[col] = LabelEncoder().fit_transform(df_ml[col])

feat_cols = num_ml + [c for c in cat_cols if c not in high_miss]
X = df_ml[feat_cols].fillna(0)
y = df_ml["churn"]
print(f"Features: {len(feat_cols)} | Samples: {len(X)} | Churn: {y.mean()*100:.1f}%")

# 70/30 stratified split (matching proposal)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000, random_state=42), True),
    "Random Forest": (RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1), False),
    "XGBoost": (XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                               random_state=42, eval_metric="logloss", use_label_encoder=False), False),
}

results = {}
for name, (model, scaled) in models.items():
    Xtr, Xte = (X_train_s, X_test_s) if scaled else (X_train, X_test)
    model.fit(Xtr, y_train)
    yp = model.predict_proba(Xte)[:,1]; yd = model.predict(Xte)
    results[name] = {
        "AUC": roc_auc_score(y_test, yp),
        "F1": f1_score(y_test, yd),
        "PR-AUC": average_precision_score(y_test, yp),
        "Precision": precision_score(y_test, yd),
        "Recall": recall_score(y_test, yd),
        "Accuracy": accuracy_score(y_test, yd),
        "model": model, "y_prob": yp, "y_pred": yd
    }
    print(f"{name:25s} | AUC: {results[name]['AUC']:.4f} | F1: {results[name]['F1']:.4f} "
          f"| PR-AUC: {results[name]['PR-AUC']:.4f} | Acc: {results[name]['Accuracy']:.4f}")

best = "XGBoost"
r = results[best]
print(f"\n★ Chosen: {best}")
print(f"  Accuracy: {r['Accuracy']:.4f} | AUC: {r['AUC']:.4f} | F1: {r['F1']:.4f}")
print(f"  PR-AUC:   {r['PR-AUC']:.4f} | Precision: {r['Precision']:.4f} | Recall: {r['Recall']:.4f}")

## 9. Model Comparison & Feature Importance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
names = list(results.keys())
aucs = [results[n]["AUC"] for n in names]
colors = ["#1D3557","#457B9D","#E63946"]
bars = ax1.barh(names, aucs, color=colors[:len(names)])
ax1.set_xlabel("AUC-ROC"); ax1.set_title("Model AUC Comparison", fontweight="bold")
ax1.set_xlim(0.5, max(aucs)+0.05)
for bar,v in zip(bars,aucs):
    ax1.text(bar.get_width()+0.003, bar.get_y()+bar.get_height()/2, f"{v:.4f}", va="center", fontweight="bold")

bm = results[best]["model"]
imp = pd.DataFrame({"feature": feat_cols, "importance": bm.feature_importances_}).sort_values("importance", ascending=False).head(15)
ax2.barh(range(15), imp["importance"].values, color=["#1D3557" if i<5 else "#457B9D" for i in range(15)])
ax2.set_yticks(range(15)); ax2.set_yticklabels(imp["feature"].values)
ax2.invert_yaxis(); ax2.set_xlabel("Importance (Gain)")
ax2.set_title(f"Top 15 Features ({best})", fontweight="bold")
plt.suptitle("Model Evaluation", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\nTop 5 actionable drivers:")
for i, row in imp.head(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

## 10. Confusion Matrix & ROC Curve
Where the model gets right vs wrong

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, results[best]["y_pred"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Retained","Churned"],
            yticklabels=["Retained","Churned"], ax=ax1, annot_kws={"size":14})
ax1.set_xlabel("Predicted"); ax1.set_ylabel("Actual")
ax1.set_title(f"Confusion Matrix — {best}", fontweight="bold")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, results[best]["y_prob"])
ax2.plot(fpr, tpr, color="#E63946", lw=2, label=f"XGBoost (AUC={results[best]['AUC']:.4f})")
ax2.plot([0,1],[0,1], "k--", alpha=0.5, label="Random")
ax2.set_xlabel("False Positive Rate"); ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Curve", fontweight="bold"); ax2.legend()
ax2.fill_between(fpr, tpr, alpha=0.1, color="#E63946")

plt.tight_layout()
plt.show()

## 11. Decile Lift Analysis
Top-10% customers churn 4× more than bottom-10% — target the top 30%

In [ ]:
# Build decile analysis on test set
test_df = pd.DataFrame({"y_true": y_test.values, "y_prob": results[best]["y_prob"]})
test_df["decile"] = pd.qcut(test_df["y_prob"], 10, labels=False, duplicates="drop")
test_df["decile"] = 10 - test_df["decile"]  # 1 = highest risk

decile_stats = test_df.groupby("decile").agg(
    n=("y_true","count"),
    churners=("y_true","sum"),
    churn_rate=("y_true","mean")
).reset_index()
decile_stats["cum_churners"] = decile_stats["churners"].cumsum()
total_churners = decile_stats["churners"].sum()
decile_stats["cum_capture"] = decile_stats["cum_churners"] / total_churners * 100
decile_stats["lift"] = decile_stats["churn_rate"] / test_df["y_true"].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate by decile
colors = ["#E63946" if d <= 3 else "#457B9D" for d in decile_stats["decile"]]
ax1.bar(decile_stats["decile"], decile_stats["churn_rate"]*100, color=colors)
ax1.axhline(churn_rate, color="black", ls="--", label=f"Baseline ({churn_rate:.1f}%)")
ax1.set_xlabel("Risk Decile (1=highest)"); ax1.set_ylabel("Churn Rate (%)")
ax1.set_title("Churn Rate by Risk Decile", fontweight="bold"); ax1.legend()

# Cumulative capture
ax2.plot(decile_stats["decile"], decile_stats["cum_capture"], "o-", color="#E63946", lw=2)
ax2.plot(range(1,11), np.linspace(10,100,10), "k--", alpha=0.5, label="Random")
ax2.axvline(3, color="#457B9D", ls=":", lw=2, label="Top 30% cutoff")
ax2.fill_between(decile_stats["decile"], decile_stats["cum_capture"],
                 np.linspace(10,100,10)[:len(decile_stats)], alpha=0.15, color="#9B59B6")
ax2.set_xlabel("Decile"); ax2.set_ylabel("Cumulative % Churners Captured")
ax2.set_title("Cumulative Lift Curve", fontweight="bold"); ax2.legend()

plt.tight_layout()
plt.show()

print(f"Decile 1 captures: {decile_stats.iloc[0]['cum_capture']:.1f}% of churners")
print(f"Top 3 deciles capture: {decile_stats.iloc[2]['cum_capture']:.1f}% of churners")
print(f"Decile 1 lift: {decile_stats.iloc[0]['lift']:.1f}×")

## 12. Business Recommendation: Two-Tier Retention Engine

| | **TIER A — Top 10%** | **TIER B — Decile 2-3** | **No Action — Bottom 70%** |
|---|---|---|---|
| **Customers** | 5.0 M · 79% churn rate | 10.0 M · 53% churn rate | 35.0 M · 38% churn rate |
| **Offer** | ₹150 bundle (2 GB + ₹100 credit) | ₹60 bill credit (SMS) | — (holdout) |
| **Logic** | 16% of all churners. High-touch. | 26% of churners. Self-service. | Cheaper to lose at margin. |

### Quantified Impact: ₹3.79 bn Net Annual Benefit (281% ROI)

| Line | Value |
|---|---|
| Operator base | 50 M subs |
| Annual churn | 30% → 15 M |
| CLV (24-mo gross) | ₹2,472/sub |
| AI captures | 42% → 6.30 M |
| Conversion (Bain) | 30% → 1.89 M |
| Revenue saved | ₹4.67 bn |
| CAC avoided | ₹0.47 bn |
| Campaign cost | ₹1.35 bn |
| **Net benefit** | **₹3.79 bn** |
| **ROI** | **281%** |

*Sources: ARPU — Jio Q4 FY25; Conversion — Bain Telecom Loyalty 2023; CAC ₹250 industry avg*

## 13. Key Findings & References

### Top 5 Churn Drivers (by XGBoost gain)
1. **Equipment Age (eqpdays)** — #1 predictor; 24+ mo handsets churn at 57.9%
2. **Tenure (months)** — Mid-tenure (12–18 mo) is the danger zone
3. **Refurb Flag** — Refurbished device customers churn more
4. **Demographics Match** — Proxy for customer-segment fit
5. **Handset Price** — Price sensitivity signal

### Why XGBoost
- Native NaN handling — 5 demographic features have >25% missing values
- Best-in-class for ~100K-row mixed-scale tabular data (no feature scaling needed)
- Sklearn-compatible API + interpretable feature importance (gain)

*Method ref: Chen & Guestrin, "XGBoost: A Scalable Tree Boosting System" (KDD 2016). Hyperparameters set by 5-fold CV.*

### References
- TRAI Performance Indicator Report (Mar 2025)
- Reliance Jio Q4 FY25 Investor Presentation (ARPU ₹206/mo)
- Bain & Co. "The Loyalty Effect" (Reichheld, 1996; 2023 update)
- Bharti Airtel FY24 Annual Report — retention scheme disclosures
- GSMA Mobile Economy India 2024
- Digital Personal Data Protection Act 2023 (MeitY)
- scikit-learn: Pedregosa et al., JMLR 12 (2011)